<a href="https://colab.research.google.com/github/zencolab/WhatDreamsCost-ComfyUI/blob/main/MiniMax_H3_turbo4%2B%E5%AF%BC%E6%BC%94%E5%8F%B0%E7%BB%BC%E5%90%88%E7%89%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MiniMax-H3 导演台全能工作流 · Colab Pro 运行版 **v4**

把 `MiniMax+H3+导演台全能工作流.json` 在 **Google Colab Pro** 上直接跑起来。
节点、权重、文件名、素材、加速、访问地址全部自动处理。

仓库里跟它配套的两个文件，Cell 1 / Cell 2 会自己拉，不用管：

| 文件 | 作用 |
| --- | --- |
| `example_workflows/minimax_h3_director_full.json` | 工作流本体，要改工作流改这个 |
| `h3_lib.py` | 公共小工具（`sh` / `log` / `wf_*` / 外网通道） |

## 执行顺序

**Cell 1 → 8 依次跑一遍。** 之后只是重启界面的话，只需 **Cell 1 + Cell 8**。

| Cell | 做什么 | 耗时 |
| --- | --- | --- |
| 1 | 配置 + 环境自检（GPU / 内存 / 磁盘） | 秒 |
| 2 | 释放内嵌 JSON + 反推需要的节点 / 模型 / 素材 | 秒 |
| 3 | ComfyUI master + torch 探测 | 2–4 分钟 |
| 4 | 导演台插件等自定义节点 | 1–2 分钟 |
| 5 | 下模型（fl2va + ref2va 两套底模，约 76 GB，断点续传） | 15–35 分钟 |
| 6 | 改写工作流并装进 ComfyUI | 秒 |
| 7 | 上传首尾帧素材 | 看图片大小 |
| 8 | 启动 + 开通道 + 体检 | 1–3 分钟 |

## Colab Pro 运行时选哪个

菜单 → **修改 → 笔记本设置**：

| 运行时 | 显存 | 能不能跑 | 自动策略 |
| --- | --- | --- | --- |
| **A100 + 高 RAM** | 40 GB | 推荐 | 普通显存模式 + `--cache-none` |
| **L4** | 22.5 GB | 可以，慢 3 倍 | `--lowvram` + 分段清显存，建议降到 0.3 MP |
| **T4** | 16 GB | 不推荐 | `--novram`，大概率 OOM 或慢到不可用 |
| CPU | — | 不行 | Cell 1 直接抦下 |

磁盘：两套底模 42 GB + 文本编码器 27 GB + VAE，共 ≈ 76 GB，A100 运行时给的盘够用；只想跑 fl2v 就把 Cell 1 的 `download_both` 改成 `False`，省 21 GB。

另外：**一定要选高 RAM**，21 GB 底模 + 27 GB 文本编码器换出时会吃掉 50 GB 以上系统内存。


In [ ]:
# ==========================================================
# Cell 1: 配置 + Colab 环境自检
#   每次连接运行时都必须先跑这一格（重启界面也只要 Cell 1 + Cell 8）
# ==========================================================
import json, os, re, shutil, subprocess, sys, time
from importlib.util import find_spec

ROOT = "/content"                       # Colab 固定盘符；本地调试可改
IN_COLAB = find_spec("google.colab") is not None

CFG = {
    "root": ROOT,
    "comfy_dir": ROOT + "/ComfyUI",
    "hf_home": ROOT + "/hf_cache",       # 和 models/ 同盘，硬链接才不会占两份
    "workflow_json": ROOT + "/workflow.json",
    "workflow_name": "MiniMaxH3_导演台全能工作流",

    # --- 权重：留空 = Cell 5 按 GPU 架构 + 剩余磁盘自动选 ---
    "repo": "Comfy-Org/MiniMax-H3",
    "dit_file": "",
    "te_file": "",
    "allow_full_dit": True,    # 已开启：Turbo LoRA 需要 34GB 非剪枝 DiT   # True = 允许下 34GB 非剪枝 DiT（只有挂 Turbo LoRA 才需要）
    #   fl2v/t2v/i2v 走 fl2va 底模，r2v/v2v/rv2v 走 ref2va 底模，是两个文件。
    #   默认两套都下（多 ≈21GB），以后在导演台里换任务不用重新下模型。
    "download_both": True,

    # --- Turbo 4 步加速 LoRA（需要非剪枝 DiT，会自动插一个 LoraLoaderModelOnly 节点）---
    "use_turbo_lora": True,    # 已开启：4 步采样，约 4~5 倍提速，且 bf16 全量模型绕开 A100 int8 convrot 慢路径
    "lora_repo": "larryvrh/MiniMax-H3-Turbo-Lora",
    "lora_src": "minimax_h3_turbo_v4_step600_ema.safetensors",   # v4-600 EMA：当前最强检查点（旧文件名在 HF 仓库已不存在）
    "lora_out": "minimax_h3_turbo_v4_step600_ema.safetensors",   # 专用节点直接读 HF 原始格式，不改名

    # --- 外网访问方式 ---
    #   "frp"        : 连你自己的 frps，TCP 直连不绕边缘节点，延迟最低（默认）
    #   "colab"      : Colab 自带端口代理开新窗口，零配置
    #   "none"       : 只在本机监听
    "tunnel": "frp",
    "local_port": 8188,
    "frp_host": "usoren.usdream.dpdns.org",
    "frp_port": 7000,
    "frp_token": "",           # frps.toml 里配了 auth.token 就填上，否则留空
    "remote_port": 8091,       # 必须在 frps 的 allowPorts 范围内
    "frp_ver": "0.56.0",
    "tunnel_fallback": True,   # frp 没连上就自动改用 Colab 端口代理，不至于白启动

    # --- Google Drive：模型/成片落盘，跨会话复用（Drive 读盘慢，按需开）---
    "use_drive": False,
    "drive_dir": "/content/drive/MyDrive/minimax_h3",
    "save_outputs_to_drive": False,

    # --- 加速模块（对应导演台 2026-08-07 新增的「加速版」示例工作流）---
    #   UNETLoader -> PathchSageAttentionKJ
    #              -> MiniMaxH3MemoryEfficientSageAttentionPatch -> 导演台
    #   两个节点都来自 KJNodes，都要 SageAttention 2 的 CUDA 内核。
    #   PyPI 上只有 1.x（没内核），Linux 也没有现成轮子，Cell 4 会用 nvcc 现编。
    "accel_sage": False,   # 已停用 Sage 模块：A100 上收益小，改用 Turbo LoRA 专用节点
    "accel_steps": 0,          # >0 就把导演台 steps 改成这个值（加速版示例是 20）
    "sage_wheel_cache": True,  # 编好的 whl 存进 Drive，下次开机秒装
    "fast_fp16_accum": True,   # 给 ComfyUI 加 --fast fp16_accumulation（sm_80+）

    # --- 其他 ---
    "sage_mode": "skip",   # 不再现编 SageAttention，省 5~12 分钟       # auto = 有 nvcc 就现编 SageAttention 2；skip = 不装、也不插加速节点
    "install_manager": True,
    "vram_policy": "auto",     # auto / highvram / lowvram / novram
    "extra_args": "",          # 额外启动参数，一般留空
}
CFG["frp_dir"] = ROOT + "/frp_%s_linux_amd64" % CFG["frp_ver"]
CFG["cfg_path"] = ROOT + "/h3_cfg.json"

os.makedirs(ROOT, exist_ok=True)
os.makedirs(CFG["hf_home"], exist_ok=True)
os.environ["HF_HOME"] = CFG["hf_home"]
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"   # Xet 后端已够快，两者同开会打架


# ---------- 公共小工具：sh / log / pip / wf_* / timeline_* / 外网通道 ----------
#   这些函数几乎不用改，放在仓库的 h3_lib.py 里，notebook 只管配置和流程。
#   exec 进全局命名空间，所以后面所有格子的用法与写在本格里完全一样。
LIB = ROOT + "/h3_lib.py"
if not os.path.exists(LIB):
    subprocess.run("wget -q -O '%s' https://raw.githubusercontent.com/zencolab/"
                   "WhatDreamsCost-ComfyUI/main/h3_lib.py" % LIB, shell=True)
exec(open(LIB, encoding="utf-8").read(), globals())

# ---------- GPU：只问 nvidia-smi，不 import torch ----------
#   在内核里 import torch 会白占 1~2GB 显存，还可能触发
#   "Only a single TORCH_LIBRARY can be used to register the namespace triton"
gpu_name, vram_gb, cap = "", 0, None
q = sh("nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader",
       check=False, quiet=True).strip()
if q and "," in q:
    parts = [p.strip() for p in q.splitlines()[0].split(",")]
    gpu_name = parts[0]
    m = re.search(r"(\d+)", parts[1])
    vram_gb = round(int(m.group(1)) / 1024) if m else 0
    if len(parts) > 2 and re.match(r"^\d+\.\d+$", parts[2]):
        cap = [int(x) for x in parts[2].split(".")]

CFG["gpu_name"], CFG["vram_gb"] = gpu_name, vram_gb
if cap:
    CFG["cap"], CFG["sm"] = cap, cap[0] * 10 + cap[1]

# ---------- 运行时档位判定 ----------
if not gpu_name:
    CFG["tier"] = "cpu"
elif vram_gb >= 38:
    CFG["tier"] = "a100"      # A100 40G / H100，最舒服
elif vram_gb >= 20:
    CFG["tier"] = "l4"        # L4 22.5G，要 --lowvram
else:
    CFG["tier"] = "t4"        # T4 16G，能不能跑要看运气

BAR = "=" * 64
print(BAR)
print("Colab 环境自检")
print(BAR)
log("运行环境   : %s" % ("Google Colab" if IN_COLAB else "非 Colab（本地/其他）"))
log("Python     : %s" % sys.version.split()[0])
log("GPU        : %s | 显存 %d GB%s"
    % (gpu_name or "未检测到", vram_gb,
       " | sm_%d" % CFG["sm"] if CFG.get("sm") else ""))
log("系统内存   : %.0f GB" % ram_gb())
log("可用磁盘   : %.0f GB" % free_gb())

if CFG["tier"] == "cpu":
    raise RuntimeError(
        "没有 GPU。菜单 → 修改 → 笔记本设置 → 硬件加速器选 GPU（Colab Pro 建议 A100 + 高 RAM）")
if CFG["tier"] == "a100":
    log("A100 档：21GB DiT + 27GB 文本编码器靠自动换进换出，正常模式即可")
elif CFG["tier"] == "l4":
    log("L4 档（22.5GB）：会自动加 --lowvram，速度约为 A100 的 1/3，建议把分辨率降到 0.3MP", "!")
else:
    log("T4 档（16GB）：无 bf16 硬件支持，H3 极易 OOM 或慢到不可用。"
        "Colab Pro 请在「修改 → 笔记本设置」里换 A100 / L4", "!")
if ram_gb() < 40:
    log("系统内存 < 40GB：模型换出时会挤爆内存，请把运行时改成「高 RAM」", "!")

# ---------- 可选：挂 Drive ----------
if CFG["use_drive"] and IN_COLAB:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    os.makedirs(CFG["drive_dir"], exist_ok=True)
    log("Drive 已挂载：" + CFG["drive_dir"])

# ---------- 可选：HF Token（左侧 🔑 Secrets）----------
if IN_COLAB:
    try:
        from google.colab import userdata
        tok = userdata.get("HF_TOKEN")
        if tok:
            os.environ["HF_TOKEN"] = tok
            log("已读取 HF_TOKEN")
    except Exception:
        log("未配置 HF_TOKEN（公开仓库不影响下载）")

save_cfg()
print(BAR)
log("配置已写入 " + CFG["cfg_path"])


In [ ]:
# ==========================================================
# Cell 2: 取工作流 JSON + 反推运行需求
#   工作流跟 notebook 一起放在仓库 example_workflows/ 里，要改工作流直接改那个文件，
#   不用再往 notebook 里塞 base64。想用自己的：把 WF_URL 清空，运行本格后手动上传。
#   读写小工具（wf_load / wf_find / director / timeline_*）都在 Cell 1，这里不重复定义。
# ==========================================================
WF_URL = ("https://raw.githubusercontent.com/zencolab/WhatDreamsCost-ComfyUI/"
          "main/example_workflows/minimax_h3_director_full.json")
WF_PATH = CFG["workflow_json"]

if not os.path.exists(WF_PATH) and WF_URL:
    sh("wget -q -O '%s' '%s'" % (WF_PATH, WF_URL), check=False, quiet=True)
    if os.path.exists(WF_PATH) and os.path.getsize(WF_PATH) < 2000:
        os.remove(WF_PATH)                     # 404 页面也会落盘，按大小判掉
    log("已拉取工作流 -> " + WF_PATH if os.path.exists(WF_PATH)
        else "工作流拉取失败，改为手动上传", "*" if os.path.exists(WF_PATH) else "!")
if not os.path.exists(WF_PATH):
    from google.colab import files
    shutil.move(list(files.upload().keys())[0], WF_PATH)
    log("已接收上传的工作流 -> " + WF_PATH)

# ---------- 解析本工作流 ----------
wf = wf_load()
CFG["wf_types"] = wf_types(wf)
d = director(wf)
tl, _i = timeline_get(d) if d else (None, -1)

print(BAR)
log("节点类型   : " + ", ".join(CFG["wf_types"]))
log("模型文件   :")
for _n, _i2, sub, fn in wf_loaders(wf):
    log("   models/%-16s %s   (%s)" % (sub + "/", fn, _n.get("title") or _n["type"]))

if d:
    task = (d["widgets_values"] or [""])[0]
    CFG["task_type"] = task
    fam = "ref2va" if re.match(r"^(r2v|v2v|rv2v)", str(task)) else "fl2va"
    CFG["model_family"] = fam
    log("导演台任务 : %s  -> 需要 %s 系底模" % (task, fam))

if tl:
    CFG["assets"] = timeline_assets(tl)
    o = tl.get("output", {})
    log("输出规格   : %sx%s | %s 帧 | %s fps | %d 段"
        % (o.get("width"), o.get("height"), tl.get("totalFrames"),
           tl.get("frameRate"), len(tl.get("segments", []))))
    log("需要素材   : %d 个（Cell 7 上传）" % len(CFG["assets"]))
    for a in CFG["assets"]:
        log("   " + a)
else:
    CFG["assets"] = []

save_cfg()
print(BAR)


In [ ]:
# ==========================================================
# Cell 3: ComfyUI 本体（master）+ torch 探测
#   MiniMax H3 的核心节点来自 PR #15224，必须用 master 最新提交
#   全程不在内核里 import torch，一律丢子进程探测
# ==========================================================
from importlib.metadata import PackageNotFoundError
from importlib.metadata import version as pkg_version

COMFY = CFG["comfy_dir"]

if not os.path.exists(COMFY):
    log("clone ComfyUI ...")
    sh("git clone --depth 1 https://github.com/comfyanonymous/ComfyUI " + COMFY)
else:
    log("更新 ComfyUI ...")
    sh("git fetch --depth 1 origin master && git checkout -q master "
       "&& git reset --hard -q origin/master", cwd=COMFY)
log("当前提交: " + sh("git log -1 --format='%h %ad %s' --date=short",
                      cwd=COMFY, quiet=True).strip())


def torch_on_disk():
    try:
        return pkg_version("torch")
    except PackageNotFoundError:
        return None


_before = torch_on_disk()
log("安装 requirements（装完校验 torch 有没有被换掉）...")
pip("-r %s/requirements.txt huggingface_hub hf_xet" % COMFY)
_after = torch_on_disk()
if _before and _after and _before != _after:
    log("torch 被改动：%s -> %s；若后面报 kernel image 错误，"
        "执行 pip install -q torch==%s 并重启运行时" % (_before, _after, _before), "!")

# ---------- 子进程探测 ----------
PROBE = "\n".join([
    "import json, torch",
    "info = {'torch': torch.__version__, 'cuda': torch.version.cuda,",
    "        'avail': torch.cuda.is_available()}",
    "if info['avail']:",
    "    p = torch.cuda.get_device_properties(0)",
    "    info['name'] = torch.cuda.get_device_name(0)",
    "    info['cap'] = list(torch.cuda.get_device_capability(0))",
    "    info['vram'] = round(p.total_memory / 1024 ** 3)",
    "    info['archs'] = torch.cuda.get_arch_list()",
    "print('PROBE=' + json.dumps(info))",
])
open(ROOT + "/probe_gpu.py", "w").write(PROBE + "\n")
out = sh("%s %s/probe_gpu.py" % (sys.executable, ROOT), check=False, quiet=True)
probe = None
for line in out.splitlines():
    if line.startswith("PROBE="):
        probe = json.loads(line[6:])
if probe is None or not probe.get("avail"):
    log("子进程输出：\n" + out[-2000:], "!")
    raise RuntimeError("torch 看不到 GPU，检查运行时类型后重跑本格")

cap = tuple(probe["cap"])
CFG["cap"], CFG["sm"], CFG["vram_gb"] = list(cap), cap[0] * 10 + cap[1], probe["vram"]
CFG["can_nvfp4"] = CFG["sm"] >= 120      # nvfp4 只有 Blackwell 有原生内核
CFG["can_fp8"] = CFG["sm"] >= 89         # fp8 需要 Ada / Hopper 以上
log("torch %s | cuda %s" % (probe["torch"], probe["cuda"]))
log("GPU: %s | sm_%d | %d GB" % (probe["name"], CFG["sm"], probe["vram"]))
log("量化可用性 : nvfp4=%s  fp8=%s  int8_convrot=是"
    % ("是" if CFG["can_nvfp4"] else "否", "是" if CFG["can_fp8"] else "否"))
if not any("sm_%d" % CFG["sm"] in a for a in probe.get("archs", [])):
    log("torch 编译目标不含 sm_%d：%s，大概率会报 no kernel image"
        % (CFG["sm"], probe.get("archs")), "!")

hit = sh("grep -rl MiniMaxH3 %s/comfy_extras %s/nodes.py || true" % (COMFY, COMFY),
         check=False, quiet=True).strip()
log("ComfyUI 原生 MiniMax H3 支持已就绪" if hit
    else "没找到 MiniMax H3 原生节点，代码不够新，重跑本格", "*" if hit else "!")

save_cfg()
log("Cell 3 完成")


In [ ]:
# ==========================================================
# Cell 4: 按工作流实际用到的节点类型装插件
#   本工作流的关键缺件是 MiniMaxH3Director（导演台），v3.1 完全没装，
#   所以打开 JSON 只会看到一个红框 "Missing Node Type"。
# ==========================================================
COMFY = CFG["comfy_dir"]

# 节点类型 -> 提供它的仓库
NODE_REPOS = {
    "MiniMaxH3Director": "https://github.com/AIMixer/ComfyUI_MiniMaxH3_Director.git",
    "PathchSageAttentionKJ": "https://github.com/kijai/ComfyUI-KJNodes.git",
    "MiniMaxH3MemoryEfficientSageAttentionPatch": "https://github.com/kijai/ComfyUI-KJNodes.git",
    "VHS_LoadVideo": "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git",
    "VHS_VideoCombine": "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git",
    "MiniMaxH3DualClockSampler": "https://github.com/shuaixn/ComfyUI-MiniMaxH3DualClockSampler.git",
}

repos = []
for t in CFG.get("wf_types", []):
    r = NODE_REPOS.get(t)
    if r and r not in repos:
        repos.append(r)
        log("工作流需要 %s -> %s" % (t, r.rsplit("/", 1)[-1]))
#   加速模块的两个节点不在内嵌工作流里，wf_types 反推不到，这里显式补上 KJNodes
if CFG.get("accel_sage") and CFG["sage_mode"] != "skip":
    kj = NODE_REPOS["PathchSageAttentionKJ"]
    if kj not in repos:
        repos.append(kj)
        log("加速模块需要 KJNodes -> ComfyUI-KJNodes")
if CFG["use_turbo_lora"]:
    #   Turbo LoRA 专用节点（MiniMaxH3TurboLoRA）：支持剪枝/非剪枝底模，运行时回注 adaln
    repos.append("https://github.com/Larryvrh/ComfyUI-MiniMax-H3-Turbo.git")
if CFG["install_manager"]:
    repos.append("https://github.com/ltdrdata/ComfyUI-Manager.git")

cn_dir = os.path.join(COMFY, "custom_nodes")
os.makedirs(cn_dir, exist_ok=True)

for repo in dict.fromkeys(repos):
    name = repo.rstrip("/").split("/")[-1].replace(".git", "")
    path = os.path.join(cn_dir, name)
    if os.path.exists(path):
        log("更新 " + name)
        sh("git pull -q", cwd=path, check=False, quiet=True)
    else:
        log("安装 " + name)
        sh("git clone --depth 1 -q %s %s" % (repo, path))
    req = os.path.join(path, "requirements.txt")
    if os.path.exists(req):
        # 过滤掉会顺手升/降 torch 的行，保住 Colab 原版
        skip = ("torch", "torchvision", "torchaudio", "triton", "#")
        keep = [l for l in open(req)
                if l.strip() and not l.lower().lstrip().startswith(skip)]
        if keep:
            open("/tmp/req.txt", "w").writelines(keep)
            pip("-r /tmp/req.txt")

# ---------- SageAttention 2（加速模块的依赖，Linux 只能现编）----------
#   PathchSageAttentionKJ 把注意力后端换成 sageattn；
#   MiniMaxH3MemoryEfficientSageAttentionPatch 换掉 H3 自注意力实现以压低峰值显存。
#   两个节点都要 SageAttention 2 的 CUDA 内核，而：
#     · PyPI 上只有 1.x（纯 Triton，没有 _qattn/_fused），`pip install sageattention==2.2.0` 必然 404；
#     · 官方现成 whl 只有 Windows 版，Linux 只能用 nvcc 自己编。
#   Colab 自带 nvcc 12.8，只编本机一种架构：A100(sm_80) 约 5~12 分钟，编完缓存到 Drive。
SAGE_REPO = "https://github.com/thu-ml/SageAttention.git"
SAGE_SRC = ROOT + "/SageAttention"
SAGE_ARCH = {80: "sm80", 86: "sm80", 89: "sm89", 90: "sm90", 120: "sm120"}


def sage_probe(kernel=True):
    """必须在 ROOT 下探测：在源码目录里 import 会误报 circular import: _fused。"""
    code = "import sageattention as s;"
    if kernel:
        ext = SAGE_ARCH.get(CFG.get("sm"), "sm80")
        code += "import sageattention._qattn_%s, sageattention._fused;" % ext
    code += "print('SAGE_OK=' + getattr(s, '__version__', '1.x'))"
    out = sh('%s -c "%s"' % (sys.executable, code), cwd=ROOT, check=False, quiet=True)
    for line in out.splitlines():
        if line.startswith("SAGE_OK="):
            return line[8:].strip()
    return None


def sage_cache():
    """Drive 上的轮子缓存目录 + 命中的文件名（按 python 版本和算力打标签）。"""
    if not (CFG["use_drive"] and CFG.get("sage_wheel_cache")
            and os.path.isdir(CFG["drive_dir"])):
        return None
    d = CFG["drive_dir"] + "/wheels"
    os.makedirs(d, exist_ok=True)
    tag = "cp%d%d-sm%s" % (sys.version_info[0], sys.version_info[1], CFG.get("sm"))
    hit = [f for f in sorted(os.listdir(d)) if f.endswith(".whl") and tag in f]
    return d, (hit[0] if hit else None)


def sage_build():
    sm = CFG.get("sm") or 0
    if sm not in SAGE_ARCH:
        log("sm_%d 不在 SageAttention 支持列表（80/86/89/90/120），不编" % sm, "!")
        return
    nvcc = shutil.which("nvcc") or "/usr/local/cuda/bin/nvcc"
    if not os.path.exists(nvcc):
        log("找不到 nvcc，编不了 SageAttention 2（换带 CUDA 工具链的 GPU 运行时）", "!")
        return
    cache = sage_cache()
    if cache and cache[1]:
        log("命中 Drive 轮子缓存 " + cache[1])
        pip("--no-deps --force-reinstall '%s/%s'" % (cache[0], cache[1]))
        return
    log("编译 SageAttention 2（sm_%d，只编这一种架构，约 5~12 分钟）..." % sm)
    shutil.rmtree(SAGE_SRC, ignore_errors=True)
    sh("git clone --depth 1 -q %s %s" % (SAGE_REPO, SAGE_SRC))
    out = "/tmp/sage_whl"
    shutil.rmtree(out, ignore_errors=True)
    env = ('CUDA_HOME=%s TORCH_CUDA_ARCH_LIST="%.1f" EXT_PARALLEL=4 '
           'NVCC_APPEND_FLAGS="--threads 8" MAX_JOBS=%d'
           % (os.path.dirname(os.path.dirname(nvcc)), sm / 10.0,
              max(2, (os.cpu_count() or 4) // 2)))
    sh("%s %s -m pip wheel . --no-deps --no-build-isolation -w %s"
       % (env, sys.executable, out), cwd=SAGE_SRC, check=False)
    whl = [f for f in os.listdir(out) if f.endswith(".whl")] if os.path.isdir(out) else []
    if not whl:
        log("SageAttention 编译失败（日志见上），加速节点会以旁路状态插入", "!")
        return
    pip("--no-deps --force-reinstall '%s/%s'" % (out, whl[0]))
    if cache:
        dst = "%s/%s__cp%d%d-sm%d.whl" % (cache[0], whl[0][:-4],
                                          sys.version_info[0], sys.version_info[1], sm)
        shutil.copy(os.path.join(out, whl[0]), dst)
        log("轮子已缓存到 " + dst)
    shutil.rmtree(SAGE_SRC, ignore_errors=True)   # 留着会让 import 走源码目录


if CFG["sage_mode"] == "skip":
    CFG["sage_ok"], CFG["sage_ver"], CFG["sage_kernel"] = False, "", ""
    log("sage_mode=skip：不装 SageAttention，Cell 6 也不会插加速节点")
else:
    ver = sage_probe(False)
    if ver and not sage_probe(True):
        log("已装的 sageattention %s 没有本机 CUDA 内核（PyPI 的 1.x 就这样），卸掉重编" % ver, "!")
        sh("%s -m pip uninstall -y -q sageattention" % sys.executable, check=False, quiet=True)
        ver = None
    elif ver:
        log("SageAttention %s 已就绪，跳过编译" % ver)
    if not ver:
        sage_build()
        ver = sage_probe(True)
    CFG["sage_ok"], CFG["sage_ver"] = bool(ver), ver or ""
    CFG["sage_kernel"] = ("sageattn_qk_int8_pv_fp16_cuda" if (CFG.get("sm") or 0) < 89
                          else "sageattn_qk_int8_pv_fp8_cuda++")
    if ver:
        log("SageAttention %s 就绪，内核 %s" % (ver, CFG["sage_kernel"]))
        if (CFG.get("sm") or 0) < 89:
            log("sm_%d 没有 fp8 张量核：KJ 节点下拉框里选 fp8 / sageattn3 一定报"
                "「执行失败」，Cell 6 已自动填 fp16 内核" % CFG.get("sm", 0), "!")
    else:
        log("SageAttention 不可用，加速节点会以旁路状态插入，不影响正常出片", "!")

# ---------- ComfyUI-Manager 离线模式：避免启动时卡在联网拉节点列表 ----------
import configparser
for p in (COMFY + "/user/__manager/config.ini",
          COMFY + "/user/default/ComfyUI-Manager/config.ini",
          COMFY + "/custom_nodes/ComfyUI-Manager/config.ini"):
    os.makedirs(os.path.dirname(p), exist_ok=True)
    cp = configparser.ConfigParser()
    if os.path.exists(p):
        cp.read(p)
    cp.setdefault("default", {})
    cp["default"]["network_mode"] = "private"
    cp.write(open(p, "w"))

save_cfg()
log("Cell 4 完成（新节点要重启 ComfyUI 才生效，即重跑 Cell 8）")


In [ ]:
# ==========================================================
# Cell 5: 模型下载（列举 HF 仓库真实文件 -> 按 GPU/磁盘自动选 -> 硬链到 models/）
#   v3.1 把文件名写死，工作流里填的又是另一个名字，启动后下拉框对不上。
#   现在下完会把实际文件名回写进 CFG，Cell 6 直接改 JSON。
#   落盘工具（link_real / safetensors_ok / fetch）在 h3_lib.py 里。
# ==========================================================
from concurrent.futures import ThreadPoolExecutor, as_completed
from huggingface_hub import HfApi, hf_hub_download

COMFY, REPO = CFG["comfy_dir"], CFG["repo"]
sm = CFG.get("sm", 80)
fam = CFG.get("model_family", "fl2va")

hf = HfApi()
try:
    info = hf.model_info(REPO, files_metadata=True)
    SIZES = {s.rfilename: (s.size or 0) for s in info.siblings}
    FILES = list(SIZES)
except Exception as e:
    log("拿不到文件大小（%r），改用文件列表" % e, "!")
    FILES, SIZES = hf.list_repo_files(REPO), {}


def pick(patterns, pool=None):
    for pat in patterns:
        for f in (pool or FILES):
            if re.search(pat, f):
                return f
    return None


def gb(f):
    return SIZES.get(f, 0) / 1024 ** 3


# ---------- 1. DiT：fl2va 和 ref2va 是两套底模 ----------
#   t2v / i2v / fl2v      -> minimax_h3_fl2va_*
#   r2v / v2v / rv2v      -> minimax_h3_ref2va_*
#   导演台里换任务类型 = 换 UNETLoader 里的文件，两套都备好就不用重下 21GB。
#   pruned 把调制权重压成查表，质量不变、体积减半；
#   只有要挂 Turbo LoRA 时必须用非剪枝版（LoRA 补的是完整 adaln_proj）。
want_full = CFG["use_turbo_lora"] or CFG["allow_full_dit"]


def dit_prefs(f):
    if want_full:
        return [r"diffusion_models/minimax_h3_%s_int8_convrot" % f,
                r"diffusion_models/minimax_h3_%s_bf16" % f]
    pref = []
    if sm >= 120:
        pref.append(r"diffusion_models/minimax_h3_%s_pruned_nvfp4" % f)   # Blackwell 专享
    pref += [r"diffusion_models/minimax_h3_%s_pruned_int8_convrot" % f,
             r"diffusion_models/minimax_h3_%s_int8_convrot" % f]
    return pref

# ---------- 2. 文本编码器 ----------
if sm >= 120:
    te_pref = [r"text_encoders/.*nvfp4", r"text_encoders/.*fp8", r"text_encoders/.*int8_convrot"]
elif sm >= 89:
    te_pref = [r"text_encoders/.*fp8", r"text_encoders/.*int8_convrot"]
else:
    te_pref = [r"text_encoders/.*int8_convrot"]      # A100 sm_80 只能走这条

fams = [fam] + ([f for f in ("fl2va", "ref2va") if f != fam]
                if CFG.get("download_both", True) else [])
dits = {}
for f in fams:
    got = pick(dit_prefs(f))
    if got:
        dits[f] = got
    else:
        log("仓库里没找到 %s 系底模，跳过" % f, "!")
if CFG["dit_file"]:
    dits[fam] = CFG["dit_file"]          # 手动指定的优先
dit = dits.get(fam)
te = CFG["te_file"] or pick(te_pref)
vae_v = pick([r"vae/.*video.*fp16", r"vae/.*video"])
vae_a = pick([r"vae/.*audio.*fp32", r"vae/.*audio"])
if not all([dit, te, vae_v, vae_a]):
    raise RuntimeError("仓库里没找齐文件：dit=%s te=%s vae=%s/%s" % (dit, te, vae_v, vae_a))

TASKS = [(REPO, dit, "diffusion_models", None),
         (REPO, te, "text_encoders", None),
         (REPO, vae_v, "vae", None),
         (REPO, vae_a, "vae", None)]
if CFG["use_turbo_lora"]:
    TASKS.append((CFG["lora_repo"], CFG["lora_src"], "loras", CFG["lora_out"]))

need = sum(gb(f) for _r, f, _d, _n in TASKS if _r == REPO) or 55
print(BAR)
log("当前任务   : %s  -> %s 系" % (CFG.get("task_type", "?"), fam))
log("DiT 主     : %s  %.1f GB" % (os.path.basename(dit), gb(dit)))
log("文本编码器 : %s  %.1f GB" % (os.path.basename(te), gb(te)))
log("VAE        : %s + %s" % (os.path.basename(vae_v), os.path.basename(vae_a)))

#   另一套底模：换任务类型时要用，磁盘够就一并下了
for f, p in [(k, v) for k, v in dits.items() if k != fam]:
    if free_gb() < need + gb(p) + 10:
        log("磁盘只剩 %.0f GB，%s 系底模（%.0f GB）这次不下；"
            "腾出空间后重跑本格即可补上" % (free_gb(), f, gb(p)), "!")
        continue
    TASKS.append((REPO, p, "diffusion_models", None))
    need += gb(p)
    log("DiT 备     : %s  %.1f GB   （r2v/v2v/rv2v 用）"
        % (os.path.basename(p), gb(p)))

log("合计 ≈ %.0f GB，当前可用 %.0f GB（HF 缓存与 models/ 同盘，硬链不翻倍）"
    % (need, free_gb()))
if free_gb() < need + 6:
    raise RuntimeError(
        "磁盘不够。处理：Cell 1 把 download_both 设为 False 只下当前任务这一套，"
        "或把 use_turbo_lora / allow_full_dit 设为 False 用剪枝版，"
        "或断开重连一个干净的 Colab 运行时")
if CFG["use_turbo_lora"] and "pruned" in dit:
    raise RuntimeError("Turbo LoRA 必须配非剪枝 DiT，否则 51 处 adaln_proj 形状不匹配")

# ---------- 3. 开跑（并行 2 路 + 断点续传）----------
t0, failed = time.time(), []
with ThreadPoolExecutor(max_workers=2) as ex:
    futs = {ex.submit(fetch, r, f, d, rn): f for r, f, d, rn in TASKS}
    for fu in as_completed(futs):
        try:
            base, msg = fu.result()
            log("%-58s %s" % (base, msg))
        except Exception as e:
            failed.append(futs[fu])
            log("下载失败 %s -> %r" % (futs[fu], e), "!")

if failed:
    log("以下文件没拿到，重跑本格可断点续传：\n  " + "\n  ".join(failed), "!")
else:
    log("全部文件就绪")

# 把实际文件名交给 Cell 6 回写 JSON（只登记真落盘了的）
DIT_DIR = os.path.join(COMFY, "models", "diffusion_models")
CFG["dit_files"] = dict((f, os.path.basename(p)) for f, p in dits.items()
                        if os.path.exists(os.path.join(DIT_DIR, os.path.basename(p))))
for f in ("fl2va", "ref2va"):
    if f in CFG["dit_files"]:
        log("%-6s 就绪：%s" % (f, CFG["dit_files"][f]))
    else:
        log("%-6s 未下载（需要时把 download_both 开着重跑 Cell 5）" % f, "!")
CFG["dit_file"], CFG["te_file"] = dit, te
CFG["vae_files"] = {"video": os.path.basename(vae_v), "audio": os.path.basename(vae_a)}
CFG["lora_ready"] = CFG["use_turbo_lora"] and not failed
save_cfg()
sh("ls -lLh %s/models/diffusion_models %s/models/text_encoders %s/models/vae"
   % (COMFY, COMFY, COMFY), check=False)
log("耗时 %.1f 分钟，剩余磁盘 %.0f GB" % ((time.time() - t0) / 60, free_gb()))
print(BAR)


In [ ]:
# ==========================================================
# Cell 6: 把工作流改成“开箱即跑”并装进 ComfyUI
#   1) 三个 loader 的文件名 -> Cell 5 实际下到的文件
#   2) 显存不到 40G 时自动打开分段清显存
#   3) 插入加速模块（Sage 补丁 x2），并按算力锁定能用的内核
#   4) 写到 user/default/workflows/，界面左侧工作流列表直接能点开
# ==========================================================
COMFY = CFG["comfy_dir"]
wf = wf_load()
changes = []

fam = CFG.get("model_family", "fl2va")
dit_files = CFG.get("dit_files") or {}
dit_base = os.path.basename(dit_files.get(fam) or CFG["dit_file"])   # 按当前任务选底模
te_base = os.path.basename(CFG["te_file"])
vae = CFG.get("vae_files", {})

for node, idx, sub, cur in wf_loaders(wf):
    new = None
    if node["type"] == "UNETLoader":
        new = dit_base
    elif node["type"] == "CLIPLoader":
        new = te_base
        wv = node["widgets_values"]
        if len(wv) > 1 and wv[1] != "minimax":
            wv[1] = "minimax"                       # type 必须是 minimax
            changes.append("CLIPLoader.type -> minimax")
    elif node["type"] == "VAELoader":
        title = (node.get("title") or "") + " " + cur
        new = vae.get("audio") if re.search(r"audio|音频", title, re.I) else vae.get("video")
    if new and new != cur:
        node["widgets_values"][idx] = new
        changes.append("%s: %s -> %s" % (node["type"], cur, new))

d = director(wf)
if d:
    tl, ti = timeline_get(d)

    # 小显存：分段之间清显存，不然第二段必 OOM
    if CFG.get("vram_gb", 0) < 38:
        j = widget_after(d, "性能", bool)
        if j >= 0 and d["widgets_values"][j] is not True:
            d["widgets_values"][j] = True
            changes.append("clear_vram_between_segments -> True")

    # Turbo LoRA：4 步
    if CFG.get("lora_ready"):
        j = widget_after(d, "高级采样", int)
        if j >= 0:
            d["widgets_values"][j] = 6
            changes.append("steps -> 6 (Turbo LoRA，4~8 范围内 6 步质量更稳)")

    if tl:
        miss = [a for a in timeline_assets(tl)
                if not os.path.exists(os.path.join(COMFY, "input", a))]
        if miss:
            changes.append("还缺 %d 个首尾帧素材，跑 Cell 7 上传" % len(miss))


def model_slot(node):
    return next((i for i, inp in enumerate(node.get("inputs", []))
                 if inp.get("name") == "model"), None)


def splice(wf, d, ntype, title, widgets, size, oname, mode=0, pos=None):
    """在上游与导演台之间插一个 MODEL 直通节点，返回改动说明。"""
    slot = model_slot(d)
    if slot is None:
        return "导演台没有 model 输入口，跳过 " + ntype
    old = next((l for l in wf["links"] if l[0] == d["inputs"][slot].get("link")), None)
    if old is None:
        return "找不到导演台的 model 连线，跳过 " + ntype
    nid = max(n["id"] for n in wf["nodes"]) + 1
    lid = max([l[0] for l in wf["links"]] + [wf.get("last_link_id", 0)]) + 1
    wf["nodes"].append({
        "id": nid, "type": ntype, "title": title,
        "pos": pos or [d["pos"][0], d["pos"][1] - 140], "size": size,
        "flags": {}, "order": 5, "mode": mode,
        "inputs": [{"name": "model", "type": "MODEL", "link": old[0]}],
        "outputs": [{"name": oname, "type": "MODEL", "links": [lid], "slot_index": 0}],
        "properties": {"Node name for S&R": ntype},
        "widgets_values": list(widgets),
    })
    old[3], old[4] = nid, 0                    # 上游 -> 新节点
    wf["links"].append([lid, nid, 0, d["id"], slot, "MODEL"])
    d["inputs"][slot]["link"] = lid            # 新节点 -> 导演台
    wf["last_link_id"] = lid
    wf["last_node_id"] = max(wf.get("last_node_id", 0), nid)
    return "插入 %s%s" % (ntype, "" if mode == 0 else "（旁路）")


# ---------- 可选：插入 Turbo LoRA 节点 ----------
if CFG.get("lora_ready") and d and not wf_find(wf, "MiniMaxH3TurboLoRA"):
    changes.append(splice(wf, d, "MiniMaxH3TurboLoRA", "MiniMax-H3 Turbo LoRA",
                          [CFG["lora_out"], 1.0, False], [360, 106], "MODEL"))

# ---------- 加速模块：UNETLoader -> Sage 补丁 x2 -> 导演台 ----------
#   对应 AIMixer/ComfyUI_MiniMaxH3_Director 的
#   example_workflows/minimax_h3_director_加速版.json（作者 2026-08-07 新增）
#   KJ 节点下拉框默认 auto，而 auto/fp8/sageattn3 在 sm_80(A100) 上都没有对应内核，
#   一选就弹「Patch Sage Attention KJ 执行失败」。这里按算力把内核写死，
#   值由 Cell 4 算好放在 CFG["sage_kernel"] 里。
SAGE_KERNEL = CFG.get("sage_kernel") or "sageattn_qk_int8_pv_fp16_cuda"
ACCEL_CHAIN = [
    ("PathchSageAttentionKJ", "Patch Sage Attention KJ",
     [SAGE_KERNEL, False], [270, 82], "MODEL"),   # False = allow_compile，别开，和加速叠加会出噪点
    ("MiniMaxH3MemoryEfficientSageAttentionPatch",
     "MiniMax H3 Mem Eff Sage Attention Patch", [], [330, 26], "model"),
]

if d and CFG.get("accel_sage") and CFG["sage_mode"] != "skip":
    mode = 0 if CFG.get("sage_ok") else 4      # 4 = 旁路，界面里 Ctrl+B 可开关
    for i, (ntype, title, widgets, size, oname) in enumerate(ACCEL_CHAIN):
        exist = wf_find(wf, ntype)
        if exist:                              # 工作流自带的：把 auto/fp8 改成能用的内核
            node = exist[0]
            if widgets and list(node.get("widgets_values") or [])[:1] != widgets[:1]:
                node["widgets_values"] = list(widgets)
                changes.append("%s 内核 -> %s" % (ntype, widgets[0]))
            if node.get("mode") == 4 and mode == 0:
                node["mode"] = 0
                changes.append("%s 取消旁路" % ntype)
            continue
        changes.append(splice(wf, d, ntype, title, widgets, size, oname, mode,
                              [d["pos"][0] + 60 * i, d["pos"][1] - 240 + 80 * i]))
    if CFG.get("accel_steps"):
        j = widget_after(d, "高级采样", int)
        if j >= 0 and d["widgets_values"][j] != CFG["accel_steps"]:
            d["widgets_values"][j] = CFG["accel_steps"]
            changes.append("steps -> %d（加速版）" % CFG["accel_steps"])

wf_save(wf)

# ---------- 装进 ComfyUI（侧栏直接能打开）----------
wf_dir = os.path.join(COMFY, "user", "default", "workflows")
os.makedirs(wf_dir, exist_ok=True)
installed = os.path.join(wf_dir, CFG["workflow_name"] + ".json")
shutil.copy2(CFG["workflow_json"], installed)
CFG["workflow_installed"] = installed

# ---------- 可选：成片存 Drive ----------
if CFG["save_outputs_to_drive"] and os.path.ismount("/content/drive"):
    out_dir = os.path.join(CFG["drive_dir"], "output")
    os.makedirs(out_dir, exist_ok=True)
    local_out = os.path.join(COMFY, "output")
    if not os.path.islink(local_out):
        shutil.rmtree(local_out, ignore_errors=True)
        os.symlink(out_dir, local_out)
    log("输出目录已指向 Drive：" + out_dir)

print(BAR)
if changes:
    log("工作流已改写：")
    for c in changes:
        log("   " + c)
else:
    log("工作流无需改写")
log("已安装到 " + installed)
if dit_files:
    log("本地底模：")
    for f in sorted(dit_files):
        log("   %-7s %s%s" % (f, dit_files[f], "   <- 当前" if f == fam else ""))
    if len(dit_files) > 1:
        log("   换 r2v/v2v/rv2v：导演台改 task_type 后，把 UNETLoader 选成 ref2va 那个文件"
            "（或重跑 Cell 2 -> 6 自动回写）")
save_cfg()
print(BAR)


In [ ]:
# ==========================================================
# Cell 7: 首尾帧素材上传
#   工作流里的图片名是作者本地的哈希名，Colab 上当然不存在，
#   不处理的话一点 Run 就报 "invalid image file"。
#   现在：你按时间线顺序上传图片 -> 自动落到 input/ 并回写文件名与真实尺寸。
#   不想现在传：把 SKIP 改成 True，之后在 ComfyUI 界面里重新选图也行。
# ==========================================================
SKIP = True    # 素材改为直接在 ComfyUI 界面上传，本格默认跳过（要用改回 False）

COMFY = CFG["comfy_dir"]
IN_DIR = os.path.join(COMFY, "input")
os.makedirs(IN_DIR, exist_ok=True)

wf = wf_load()
d = director(wf)
tl, ti = timeline_get(d) if d else (None, -1)
needed = timeline_assets(tl) if tl else []
missing = [a for a in needed if not os.path.exists(os.path.join(IN_DIR, a))]

print(BAR)
log("时间线需要 %d 个素材，缺 %d 个" % (len(needed), len(missing)))
for i, a in enumerate(needed, 1):
    log("  %d. %s  %s" % (i, a, "✓" if a not in missing else "✗ 缺失"))

if SKIP or not missing:
    log("无需上传")
elif not IN_COLAB:
    log("非 Colab 环境，请自行把文件放到 " + IN_DIR, "!")
else:
    from google.colab import files
    print("\n按时间线顺序选图（可一次多选，按文件名排序对应上面的 1..%d）：" % len(needed))
    up = files.upload()
    names = sorted(up.keys())
    if not names:
        log("没有上传任何文件", "!")
    else:
        if len(names) != len(needed):
            log("上传 %d 个 / 需要 %d 个，按顺序尽量对应" % (len(names), len(needed)), "!")
        mapping = {}
        for old, new in zip(needed, names):
            dst = os.path.join(IN_DIR, os.path.basename(new))
            shutil.move(new, dst)
            mapping[old] = os.path.basename(new)
            log("%s  ->  %s" % (new, old))
        for n in names[len(needed):]:
            shutil.move(n, os.path.join(IN_DIR, os.path.basename(n)))

        # 把时间线里的旧文件名换成新名，并把宽高改成真实尺寸
        try:
            from PIL import Image
            def wh(fn):
                p = os.path.join(IN_DIR, fn)
                with Image.open(p) as im:
                    return im.size
        except Exception:
            def wh(fn):
                return None

        def fix(obj):
            if isinstance(obj, dict):
                for k in ("imageFile", "videoFile", "audioFile", "fileName"):
                    if obj.get(k) in mapping:
                        obj[k] = mapping[obj[k]]
                        s = wh(obj[k]) if k == "imageFile" else None
                        if s:
                            obj["width"], obj["height"] = s
                for v in obj.values():
                    fix(v)
            elif isinstance(obj, list):
                for v in obj:
                    fix(v)

        fix(tl)
        timeline_set(d, tl, ti)
        wf_save(wf)
        shutil.copy2(CFG["workflow_json"], CFG["workflow_installed"])
        log("时间线已回写新文件名与尺寸")

log("input/ 目录：")
sh("ls -lh %s | tail -n 20" % IN_DIR, check=False)
print(BAR)


In [ ]:
# ==========================================================
# Cell 8: 启动 ComfyUI + 外网通道 + 工作流可运行性体检
#   重启界面只需重跑 Cell 1 + 本格
# ==========================================================
import threading

CFG = load_cfg()
COMFY, PORT = CFG["comfy_dir"], CFG["local_port"]
assert os.path.isdir(COMFY), "先跑 Cell 3"

# ---------- 1. 显存策略 ----------
#   ComfyUI 只有 --gpu-only / --highvram / --lowvram / --novram / --cpu，
#   普通模式就是一个都不传，没有 --normalvram 这个参数。
vram = CFG.get("vram_gb", 0)
policy = CFG["vram_policy"]
if policy == "auto":
    policy = "normal" if vram >= 38 else ("lowvram" if vram >= 20 else "novram")
# 原为 "--cache-none --disable-smart-memory"：每次 Run 都强制重载 21GB DiT + 27GB 文本编码器并重新编码文本，
# 交互式使用是纯粹的时间黑洞（只有做基准测试才需要 --cache-none）。
# 若去掉后 Colab 系统内存吃紧 / 进程被 OOM 杀，改用下面注释里的回退参数：
flags = ""
# flags = "--disable-pinned-memory"   # 内存回退方案：RSS 可从 ~45GB 降到 ~6GB，代价是权重换入稍慢
if policy != "normal":
    flags += " --" + policy
if CFG.get("fast_fp16_accum") and (CFG.get("sm") or 0) >= 80 and CFG.get("sage_ok"):
    flags += " --fast fp16_accumulation"       # 本版 ComfyUI 不认识的话下面会自动剔掉
flags = (flags + " " + CFG["extra_args"]).strip()

flags, dropped = sanitize(flags, supported_flags())
if dropped:
    log("本版 ComfyUI 不认识 %s，已剔除" % " ".join(dropped), "!")
log("显存策略: %s | 参数: %s" % (policy, flags or "(无)"))

# ---------- 2. 启动 ----------
for f in (ROOT + "/comfy.log", ROOT + "/frpc.log"):
    if os.path.exists(f):
        os.remove(f)
sh("pkill -f frpc || true; pkill -f 'ComfyUI/main.py' || true", check=False, quiet=True)

launch = ("python main.py --listen 127.0.0.1 --port %d --enable-cors-header '*' "
          "--preview-method auto --disable-auto-launch %s" % (PORT, flags))
log("launch: " + launch)
subprocess.Popen(launch + " > %s/comfy.log 2>&1" % ROOT, shell=True, cwd=COMFY)

ready = False
for i in range(300):
    time.sleep(2)
    txt = open(ROOT + "/comfy.log", errors="ignore").read() if os.path.exists(ROOT + "/comfy.log") else ""
    if "To see the GUI go to" in txt:
        ready = True
        break
    if "main.py: error:" in txt or ("Traceback" in txt and i > 15):
        log("启动报错，日志尾部：\n" + txt[-3000:], "!")
        break
log("ComfyUI 已就绪" if ready else "ComfyUI 未就绪", "*" if ready else "!")

# ---------- 3. 体检：节点在不在、文件名对不对、素材齐不齐 ----------
problems = []
if ready:
    wf = wf_load()
    bypassed = {n.get("type") for n in wf["nodes"] if n.get("mode") == 4}
    for t in wf_types(wf):
        if t in ("Note", "MarkdownNote", "Reroute") or t in bypassed:
            continue          # 旁路(Ctrl+B)的节点不参与体检
        try:
            info = api("/object_info/" + t)
        except Exception:
            info = {}
        if not info.get(t):
            problems.append("缺节点 %s（重跑 Cell 4，或用 Manager 搜一下）" % t)
            continue
        req = (info[t].get("input", {}).get("required") or {})
        for node, idx, sub, fn in wf_loaders(wf):
            if node["type"] != t:
                continue
            field = list(req)[idx] if len(req) > idx else None
            opts = req.get(field, [None])[0] if field else None
            if isinstance(opts, list) and fn not in opts:
                problems.append("%s 选不到 %s（可选：%s）"
                                % (t, fn, ", ".join(map(str, opts[:4])) or "空"))
    d = director(wf)
    tl, _ = timeline_get(d) if d else (None, -1)
    for a in (timeline_assets(tl) if tl else []):
        if not os.path.exists(os.path.join(COMFY, "input", a)):
            problems.append("缺素材 input/%s（跑 Cell 7 上传）" % a)

# ---------- 4. 外网通道 ----------
#   frp = TCP 直连你自己的 frps，不经过任何公共边缘节点，延迟最低
url = "http://127.0.0.1:%d" % PORT
mode = CFG["tunnel"] if ready else "none"

if mode == "frp":
    got = start_frp()
    if not got and CFG.get("tunnel_fallback"):
        log("frp 没起来，先用 Colab 端口代理兜底，修好 frps 后重跑本格即可", "!")
        got = start_colab_proxy()
    url = got or url
elif mode == "colab":
    url = start_colab_proxy() or url

# ---------- 5. keep-alive，Colab 空闲 90 分钟会断 ----------
def _keep():
    while True:
        time.sleep(300)
        print("[keep-alive]", flush=True)


threading.Thread(target=_keep, daemon=True).start()

print("\n" + BAR)
print("ComfyUI  : %s" % ("READY" if ready else "NOT READY"))
print("访问地址 : %s" % url)
print("工作流   : 左侧侧栏 Workflows -> %s" % CFG["workflow_name"])
print("底模     : %s" % os.path.basename(CFG.get("dit_file", "")))
_alt = [v for k, v in (CFG.get("dit_files") or {}).items()
        if v != os.path.basename(CFG.get("dit_file", ""))]
if _alt:
    print("备用底模 : %s（换 r2v/v2v/rv2v 时在 UNETLoader 里选它）" % ", ".join(_alt))
print("文本编码 : %s" % os.path.basename(CFG.get("te_file", "")))
print("加速     : %s" % ("SageAttention %s + %s" % (CFG.get("sage_ver"), CFG.get("sage_kernel"))
                        if CFG.get("sage_ok") else "未启用（加速节点已旁路，选中按 Ctrl+B 可开关）"))
if problems:
    print("-" * 64)
    print("体检发现 %d 个问题：" % len(problems))
    for p in problems:
        print("  ✗ " + p)
else:
    print("体检     : 节点 / 模型文件 / 素材 全部对得上，可以直接 Run")
print(BAR + "\n")
if not ready:
    print(open(ROOT + "/comfy.log", errors="ignore").read()[-3000:])

subprocess.run("tail -f %s/comfy.log" % ROOT, shell=True)


## 报错速查

| 现象 | 原因 | 处理 |
| --- | --- | --- |
| 节点变红框 `MiniMaxH3Director` | 导演台插件没装上或没重启 | 重跑 Cell 4，再重跑 Cell 8（新节点要重启服务才加载） |
| 下拉框里没有模型文件 | ComfyUI 只在启动时扫目录 | 界面右上角 Refresh，或重跑 Cell 8 |
| `[Errno 2] No such file or directory: models/vae/xxx.safetensors`，`ls` 里却看得到 | 把 HF 缓存的相对符号链接硬链过去了，悬空 | v4 已修（先 realpath）。手动清：`find /content/ComfyUI/models -xtype l -delete` 后重跑 Cell 5 |
| `invalid image file` / 首尾帧报错 | `input/` 里没有时间线引用的图 | 跑 Cell 7 上传，或在导演台面板里重新选图 |
| `CUDA out of memory` | 分辨率/帧数超了本档显存 | 导演台里把 megapixels 降到 0.3；每段 124 帧 → 85 帧；确认分段清显存已开 |
| 会话直接死掉 / 重连 | 系统内存爆了 | 运行时改成高 RAM；保持 `--cache-none`；先只跑一段验证 |
| `main.py: error: unrecognized arguments` | 自己加的 `extra_args` 本版不认识 | Cell 8 会自动剔除并提示；显存档位只有 `--gpu-only/--highvram/--lowvram/--novram/--cpu` |
| `CUDA error: no kernel image is available` | ComfyUI requirements 把 torch 换掉了 | 按 Cell 3 打印的旧版本号 `pip install -q torch==<旧版>`，然后重启运行时 |
| `Only a single TORCH_LIBRARY ... triton` | 在内核里 reload 了 torch | 不要在单元格里 `import torch`；已发生就重启运行时，只跑 Cell 1 + 8 |
| frp `port not allowed` | frps 的 `allowPorts` 不含 8091 | frps.toml 里放行，或把 `remote_port` 改成已放行的端口 |
| frp `port already used` | 上一个会话的 frpc 还挂在 frps 上 | 等 30 秒重跑 Cell 8，或换一个 `remote_port` |
| frp `login to server failed` | frps 没跑 / 7000 没开 / 域名解析不对 | 先在本地 `telnet frp_host 7000` 试一下 |
| frp `token ... doesn't match` | `frp_token` 和 frps 对不上 | Cell 1 填对 `frp_token` |
| 换了 r2v / v2v / rv2v 任务报形状错 | 这些任务要 `ref2va` 底模，你还挂着 fl2va | 两套底模默认都已在本地：把 UNETLoader 换成 `minimax_h3_ref2va_*` 那个文件即可，或重跑 Cell 2 → 6 自动回写 |
| `Patch Sage Attention KJ 执行失败` | 装的是 PyPI 的 sageattention 1.x，没有 CUDA 内核 | 重跑 Cell 4：自动卸掉 1.x，用 nvcc 现编 SageAttention 2 |
| `Could not find a version that satisfies sageattention==2.2.0` | PyPI 上只发到 1.0.6，2.x 从没上架 | 别 pip 装 2.x，Cell 4 会从源码编 |
| 下拉框选 `fp8` / `sageattn3` 必报错 | fp8 张量核要 sm_89+，A100 是 sm_80 | 只能用 `sageattn_qk_int8_pv_fp16_cuda`，Cell 6 已自动填 |
| `cannot import name '_fused'` | 在 `/content/SageAttention` 源码目录里 import | 换到 `/content` 再 import；Cell 4 编完会删源码目录 |
| `no kernel image is available` | 轮子架构和本机 GPU 对不上 | 删掉 Drive 里 `wheels/` 的缓存轮子，重跑 Cell 4 重编 |
| `nvcc: command not found` | 运行时没带 CUDA 工具链 | 换 GPU 运行时，没 nvcc 就编不了 SageAttention 2 |
| 开了加速后画面和之前不一样 | 加速走的是近似注意力（int8/fp8 量化） | 正常现象；要同 seed 严格比对就把加速模块旁路 |

---

## 这个工作流里的默认参数

| 项 | 值 | 说明 |
| --- | --- | --- |
| 任务 | `fl2v` 首尾帧生视频 | 对应 `fl2va` 底模 |
| 分辨率 | 576 × 736（3:4 竖版 0.4 MP） | A100 下很宽裕 |
| 分段 | 3 段 × 124 帧 @ 24fps | 共 372 帧，约 15.5 秒 |
| steps / sampler | 20 / `res_multistep` + `simple` | 不挂 LoRA 的正常档 |
| shift video / audio | 12 / 3 | 已验证组合，别改 |
| cfg | 1.0 | 等价 BasicGuider |

想用 **Turbo 4 步 LoRA**：Cell 1 把 `use_turbo_lora` 改 `True`。它必须配**非剪枝 34 GB 底模**（剪枝版会在 51 处报 `adaln_proj` 形状不匹配），Cell 5 会自动改下非剪枝版，Cell 6 插入 LoRA 节点并把 steps 改成 4。

---

## 存盘与提速

- `use_drive = True`：把 Drive 挂上；`save_outputs_to_drive = True` 会把 `ComfyUI/output` 指向 Drive，会话断了成片还在。
- 模型本体不建议放 Drive：27 GB 文本编码器从 Drive 读一次比重下一次还慢。
- **加速模块**：`accel_sage = True`（默认）会让 Cell 4 装好 KJNodes 并现编 SageAttention 2，Cell 6 自动把 `UNETLoader → PathchSageAttentionKJ → MiniMaxH3MemoryEfficientSageAttentionPatch → 导演台` 串起来，对应作者 2026-08-07 新增的 `minimax_h3_director_加速版.json`。
- SageAttention 2 **没有 Linux 现成轮子**（PyPI 只到 1.x，官方 whl 只有 Windows），只能 nvcc 现编：A100 首次约 5~12 分钟；`sage_wheel_cache = True` 会把 whl 存进 Drive，之后开机秒装。
- 编不出来时两个节点仍会插入但处于**旁路**状态，选中按 `Ctrl+B` 即可开关，出片不受影响。
- `fast_fp16_accum = True` 会加 `--fast fp16_accumulation`（sm_80+），和 SageAttention 叠加再快一档；`allow_compile` 保持 `False`，和 torch.compile 一起用有出噪点的报告。
- 加速是近似计算（Q/K 量化到 int8、V 到 fp8），画面与不开时会有细微差别；要严格做同 seed 比对就把它旁路掉。
- 不想要就把 `accel_sage` 设成 `False`，或把 `sage_mode` 设成 `"skip"`。
- 导演台面板里的 `Ollama` 提示词扩写需要本地 Ollama 服务，Colab 上没有，不要点那个按钮。
